# 2-4. 采样与重构

**Sampling and Reconstruction**

---

本notebook介绍模拟信号与数字信号之间的桥梁：采样定理与信号重构。从连续时间信号到离散样本，再到完美重构，采样理论是现代通信系统的基石。

## 1. 学习目标

- **理解采样定理**：奈奎斯特采样定理，理解过采样与欠采样
- **掌握抗混叠滤波器**：采样前滤除高频分量
- **理解量化**：A/D转换中的量化噪声
- **掌握信号重构**：零阶保持、内插公式完美重构条件
- **理解采样率转换**：抽取与内插的基本原理

## 2. 概念讲解

### 2.1 奈奎斯特采样定理

**定理**：如果信号x(t)的最高频率分量为B，则采样频率Fs ≥ 2B时，可以完美恢复原信号。


**混叠**：当Fs < 2B时，高频分量会折叠到低频，造成失真。

**奈奎斯特频率**：Fs/2，是无混叠的最高信号频率。

### 2.2 理想重构

理想低通滤波器可以完美重构采样信号：
$$x(t) = \sum_{n=-\infty}^{\infty} x[n] \cdot \text{sinc}\left(\frac{t - nT_s}{T_s}\right)$$


其中sinc(x) = sin(πx)/(πx)。

## 3. Python演示

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("采样与重构演示环境已准备就绪")

### 3.1 采样定理演示

In [ ]:
# 采样定理演示
def demo_sampling():
    """
    演示过采样与欠采样效果
    """
    # 原始信号：多频率正弦波
    Fs = 1000  # 采样频率
    t = np.arange(0, 0.1, 1/Fs)
    f_signal = 50  # 信号频率 50Hz
    x = np.sin(2*np.pi*f_signal*t) + 0.5*np.sin(2*np.pi*150*t)  # 50Hz + 150Hz
    
    # 过采样：Fs = 1000Hz (远大于2*150=300Hz)
    Fs_over = 1000
    t_over = np.arange(0, 0.1, 1/Fs_over)
    x_over = np.sin(2*np.pi*f_signal*t_over) + 0.5*np.sin(2*np.pi*150*t_over)
    
    # 欠采样：Fs = 200Hz (< 2*150=300Hz)
    Fs_under = 200
    t_under = np.arange(0, 0.1, 1/Fs_under)
    x_under = np.sin(2*np.pi*f_signal*t_under) + 0.5*np.sin(2*np.pi*150*t_under)
    
    # 临界采样：Fs = 300Hz (= 2*150)
    Fs_critical = 300
    t_critical = np.arange(0, 0.1, 1/Fs_critical)
    x_critical = np.sin(2*np.pi*f_signal*t_critical) + 0.5*np.sin(2*np.pi*150*t_critical)
    
    # 可视化
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    
    # 原始信号
    ax = axes[0]
    ax.plot(t*1000, x, 'b-', linewidth=1.5)
    ax.set_title('原始模拟信号 (50Hz + 150Hz)', fontsize=12)
    ax.set_ylabel('幅度', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100)
    
    # 过采样
    ax = axes[1]
    ax.plot(t_over*1000, x_over, 'b-', linewidth=1, alpha=0.7)
    ax.scatter(t_over*1000, x_over, s=20, c='red', zorder=5)
    ax.set_title(f'过采样 Fs={Fs_over}Hz (远大于2×150Hz)', fontsize=12)
    ax.set_ylabel('幅度', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100)
    
    # 临界采样
    ax = axes[2]
    ax.plot(t*1000, x, 'b-', linewidth=1, alpha=0.5, label='原始')
    ax.scatter(t_critical*1000, x_critical, s=50, c='red', zorder=5)
    ax.set_title(f'临界采样 Fs={Fs_critical}Hz (= 2×150Hz)', fontsize=12)
    ax.set_ylabel('幅度', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100)
    
    # 欠采样
    ax = axes[3]
    ax.plot(t*1000, x, 'b-', linewidth=1, alpha=0.5, label='原始')
    ax.scatter(t_under*1000, x_under, s=50, c='red', zorder=5)
    ax.set_title(f'欠采样 Fs={Fs_under}Hz (< 2×150Hz) - 发生混叠', fontsize=12)
    ax.set_xlabel('时间 (ms)', fontsize=11)
    ax.set_ylabel('幅度', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"原始信号频率: {f_signal}Hz 和 150Hz")
    print(f"奈奎斯特频率: {Fs_critical/2}Hz")
    print(f"欠采样后，150Hz信号折叠到 {abs(150 - Fs_under + Fs_under//2)} Hz")
    print(f"实际观察到: 150Hz mod 200Hz = {(150 % Fs_under)}")

demo_sampling()

### 3.2 信号重构演示

In [ ]:
# 信号重构演示
def demo_reconstruction():
    """
    演示零阶保持和一阶内插重构
    """
    # 采样信号
    Fs = 100
    t = np.arange(0, 0.5, 1/Fs)
    x = np.sin(2*np.pi*10*t) + 0.3*np.sin(2*np.pi*20*t)
    
    # 重构：零阶保持
    Fs_recon = 20  # 低采样率
    t_recon = np.arange(0, 0.5, 1/Fs_recon)
    x_recon_sample = np.sin(2*np.pi*10*t_recon) + 0.3*np.sin(2*np.pi*20*t_recon)
    
    # 零阶保持重构
    t_zoh = np.arange(0, 0.5, 1/1000)
    x_zoh = np.interp(t_zoh, t_recon, x_recon_sample)
    
    # 一阶内插重构
    from scipy.interpolate import interp1d
    f_linear = interp1d(t_recon, x_recon_sample, kind='linear')
    x_linear = f_linear(t_zoh)
    
    # 可视化
    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.plot(t*1000, x, 'k-', linewidth=2, label='原始信号', alpha=0.8)
    ax.scatter(t_recon*1000, x_recon_sample, s=80, c='blue', zorder=5, 
               label=f'采样点 (Fs={Fs_recon}Hz)')
    ax.plot(t_zoh*1000, x_zoh, 'g--', linewidth=1.5, label='零阶保持重构', alpha=0.8)
    ax.plot(t_zoh*1000, x_linear, 'r-', linewidth=1.5, label='一阶内插重构', alpha=0.8)
    
    ax.set_xlabel('时间 (ms)', fontsize=11)
    ax.set_ylabel('幅度', fontsize=11)
    ax.set_title('采样信号重构比较', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return t, x, x_zoh, x_linear

t, x, x_zoh, x_linear = demo_reconstruction()

## 4. 思考题

1. **采样定理**：如果信号包含DC分量、10Hz、100Hz分量，最小采样频率是多少？
2. **混叠分析**：一个200Hz的正弦波，如果用150Hz采样，观察到的频率是多少？
3. **重构质量**：为什么实际系统中零阶保持比理想低通更常用？
4. **量化噪声**：A/D转换的量化步长为Δ，量化噪声功率是多少？

---

## 总结

本notebook介绍了采样与重构的核心概念：
- **奈奎斯特采样定理**：Fs ≥ 2B 保证无失真恢复
- **混叠**：Fs < 2B时产生折叠失真
- **信号重构**：零阶保持和一阶内插是实际常用的方法
- **采样率转换**：抽取减少采样率，内插增加采样率